# Scoring new pages with the selector
**Ariel Lax** | Corpus2GeoRAG / CollectCorpus

New pages found one link away from the corpus (BFS, depth 1) are scored by the trained selector:
```
page ─► paragraphs ─► E5 ─► logistic regression ─► related paragraphs (p ≥ 0.7) ─► score = their share
     ─► accept (≥ 0.30 or ≥ 7 related paragraphs) / review (0.10–0.30) / reject (< 0.10)
```
### Before running
1. Put **`candidates_links10.jsonl`** (from `CollectCorpus/data/expansion/links10/candidates.jsonl`,
   renamed) in **`MyDrive/selector`**.
2. Runtime → Change runtime type → **T4 GPU**. Runtime → **Run all**, and approve the Drive connection.
3. At the end, download **`candidate_scores_links10.csv`** from `MyDrive/selector`.

Vectors are saved to Drive in chunks; if the session disconnects, **Run all** again continues.

## Setup

In [ ]:
import json, os, re, urllib.request
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (this will be slow)')

In [ ]:
LOCAL = False   # True = test run on a local folder (no Drive / GitHub)
NAME = 'links10'  # which candidate set

if LOCAL:
    WORK, MODEL_DIR = 'data/expansion/score_test', 'selector/model'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK, MODEL_DIR = '/content/drive/MyDrive/selector', 'model_repo'
    os.makedirs(MODEL_DIR, exist_ok=True)
    for f in ['selector_config.json', 'selector_logreg.joblib']:      # the committed selector, from GitHub
        urllib.request.urlretrieve('https://raw.githubusercontent.com/arielax-212/Corpus2GeoRAG/main/CollectCorpus/selector/model/' + f, f'{MODEL_DIR}/{f}')
CANDIDATES = f'{WORK}/candidates_{NAME}.jsonl'
assert os.path.exists(CANDIDATES), f'Put candidates_{NAME}.jsonl in {WORK}'
print('working folder:', WORK, '| model:', os.listdir(MODEL_DIR))

## The selector code (`build_training_set.py`, `selector.py`)

In [ ]:
MIN_CHARS = 50
SKIP_SECTIONS = {"References", "See also", "External links", "Further reading", "Notes", "Bibliography",
                 "Sources", "Citations", "Footnotes", "Explanatory notes", "Works cited"}

In [ ]:
def split_paragraphs(content):
    """[(section path, paragraph), ...] following the rules in the module docstring."""
    out, path, skip = [], [], False
    for line in content.split("\n"):
        line = line.strip()
        heading = re.match(r"^(=+)\s*(.*?)\s*=+$", line)
        if heading:
            level = len(heading.group(1)) - 1          # "== X ==" is level 1
            path = path[:level - 1] + [heading.group(2)]
            if level == 1:
                skip = heading.group(2) in SKIP_SECTIONS
            continue
        if not skip and len(line) >= MIN_CHARS:
            out.append((" > ".join(path) or "(intro)", line))
    return out

In [ ]:
E5_MODEL = "intfloat/multilingual-e5-large"
PARAGRAPH_THRESHOLD = 0.7
ACCEPT_THRESHOLD = 0.30
REJECT_THRESHOLD = 0.10
MIN_RELATED_PARAGRAPHS = 7

In [ ]:
class Selector:
    def __init__(self, encoder, model_name=E5_MODEL, paragraph_threshold=PARAGRAPH_THRESHOLD,
                 accept_threshold=ACCEPT_THRESHOLD, reject_threshold=REJECT_THRESHOLD,
                 min_related_paragraphs=MIN_RELATED_PARAGRAPHS, clf=None):
        self.encoder = encoder              # a sentence_transformers.SentenceTransformer (E5)
        self.model_name = model_name
        self.paragraph_threshold = paragraph_threshold
        self.accept_threshold = accept_threshold
        self.reject_threshold = reject_threshold
        self.min_related_paragraphs = min_related_paragraphs
        self.clf = clf

    # ---------------------------------------------------------------- vectors
    def embed(self, texts, batch_size=64, show_progress_bar=False):
        """E5 vectors; E5 expects the "query: " prefix for this kind of task."""
        return self.encoder.encode(["query: " + t for t in texts], batch_size=batch_size,
                                   normalize_embeddings=True, show_progress_bar=show_progress_bar)

    # ---------------------------------------------------------------- training
    def fit(self, paragraphs, labels, vectors=None):
        """Logistic regression on the paragraph vectors. class_weight="balanced" evens out the
        different number of paragraphs in the two groups."""
        x = self.embed(paragraphs, show_progress_bar=True) if vectors is None else vectors
        self.clf = LogisticRegression(C=10, max_iter=2000, class_weight="balanced").fit(x, labels)
        return self

    # ---------------------------------------------------------------- scoring
    def paragraph_scores(self, paragraphs, vectors=None):
        """Probability (0..1) that each paragraph is related."""
        if vectors is None:
            if not paragraphs:
                return np.array([])
            vectors = self.embed(paragraphs)
        return self.clf.predict_proba(vectors)[:, 1]

    def page_score(self, paragraph_probs, paragraph_threshold=None):
        """Share of the page's paragraphs that count as related (0..1)."""
        t = self.paragraph_threshold if paragraph_threshold is None else paragraph_threshold
        return float(np.mean(np.asarray(paragraph_probs) >= t)) if len(paragraph_probs) else 0.0

    def decision(self, score, related=0):
        """"accept", "review" or "reject" for a page score (and its number of related paragraphs)."""
        if score >= self.accept_threshold or (self.min_related_paragraphs and related >= self.min_related_paragraphs):
            return "accept"
        return "reject" if score < self.reject_threshold else "review"

    def score_page(self, text):
        """Everything for one page's raw text: {"score", "decision", "paragraphs", "related_paragraphs"}."""
        paragraphs = [p for _, p in split_paragraphs(text)]
        probs = self.paragraph_scores(paragraphs)
        score = self.page_score(probs)
        related = int((probs >= self.paragraph_threshold).sum())
        return {"score": round(score, 3), "decision": self.decision(score, related), "paragraphs": len(paragraphs),
                "related_paragraphs": related}

    # ---------------------------------------------------------------- save / load
    def save(self, folder):
        """The classifier and the settings; the E5 model itself is downloaded by name when loading."""
        folder = Path(folder)
        folder.mkdir(parents=True, exist_ok=True)
        joblib.dump(self.clf, folder / "selector_logreg.joblib")
        (folder / "selector_config.json").write_text(json.dumps({
            "model_name": self.model_name, "paragraph_threshold": self.paragraph_threshold,
            "accept_threshold": self.accept_threshold, "reject_threshold": self.reject_threshold,
            "min_related_paragraphs": self.min_related_paragraphs}, indent=2),
            encoding="utf-8")

    @classmethod
    def load(cls, folder, encoder=None):
        folder = Path(folder)
        config = json.loads((folder / "selector_config.json").read_text(encoding="utf-8"))
        if encoder is None:
            from sentence_transformers import SentenceTransformer
            encoder = SentenceTransformer(config["model_name"])
        return cls(encoder, clf=joblib.load(folder / "selector_logreg.joblib"), **config)

## 1. Load the selector and the candidate pages

In [ ]:
encoder = SentenceTransformer(E5_MODEL, device='cuda' if torch.cuda.is_available() else 'cpu')
gate = Selector.load(MODEL_DIR, encoder=encoder)
print({k: getattr(gate, k) for k in ['model_name', 'paragraph_threshold', 'accept_threshold', 'reject_threshold',
                                     'min_related_paragraphs']})

pages = [json.loads(l) for l in open(CANDIDATES, encoding='utf-8')]
para_page, para_text = [], []
for i, p in enumerate(pages):
    for _, text in split_paragraphs(p['content']):
        para_page.append(i); para_text.append(text)
para_page = np.array(para_page, dtype=int)
print(f'{len(pages)} pages, {len(para_text)} paragraphs')

## 2. E5 vectors (in chunks, saved to Drive)

In [ ]:
CHUNK = 5000
chunk_dir = Path(WORK) / f'candidate_vectors_{NAME}'
chunk_dir.mkdir(parents=True, exist_ok=True)
parts = []
for start in range(0, len(para_text), CHUNK):
    path = chunk_dir / f'{start:07d}_{len(para_text)}.npy'    # the total in the name: a new candidate file never reuses old chunks
    if not path.exists():
        np.save(path, gate.embed(para_text[start:start + CHUNK]))
        print('  vectors', min(start + CHUNK, len(para_text)), '/', len(para_text))
    parts.append(np.load(path))
vectors = np.vstack(parts) if parts else np.zeros((0, 1024))

## 3. Scores and decisions

In [ ]:
probs = gate.paragraph_scores(None, vectors=vectors) if len(vectors) else np.array([])
related = np.bincount(para_page, weights=probs >= gate.paragraph_threshold, minlength=len(pages)).astype(int)
total = np.bincount(para_page, minlength=len(pages))
score = np.divide(related, total, out=np.zeros(len(pages)), where=total > 0)
scores = pd.DataFrame({'title': [p['title'] for p in pages], 'revid': [p['revid'] for p in pages],
                       'links_from_corpus': [p.get('links_from_corpus') for p in pages],
                       'paragraphs': total, 'related_paragraphs': related, 'score': np.round(score, 3)})
scores['decision'] = [gate.decision(s, r) for s, r in zip(score, related)]
scores.to_csv(f'{WORK}/candidate_scores_{NAME}.csv', index=False)
print(scores.decision.value_counts().reindex(['accept', 'review', 'reject']).to_string())
print('saved', f'{WORK}/candidate_scores_{NAME}.csv')

In [ ]:
for d in ['accept', 'review', 'reject']:
    print(f'--- {d}: top by links from the corpus')
    display(scores[scores.decision == d].nlargest(10, 'links_from_corpus')[['title', 'links_from_corpus', 'paragraphs', 'related_paragraphs', 'score']])